# Frontier × cuOpt — GPU-accelerated portfolio frontier

**What this shows.** Frontier is a multi-objective decision engine: its default
solver evolves whole allocations with an evolutionary algorithm (pymoo NSGA-II)
and scores them against every objective. This notebook swaps in a *second*
solver for the special case where the math is a **convex quadratic program (QP)** —
mean-variance portfolio optimization — and routes it to **NVIDIA cuOpt** on the GPU.

**The decomposition (why a QP solver fits a multi-objective engine).** A
risk/return portfolio frontier is a one-parameter family: for each target return
`r`, the lowest-risk portfolio is the solution of

$$\min_{w}\; w^{\top}\Sigma w \quad\text{s.t.}\quad \mu^{\top}w \ge r,\;\; \mathbf{1}^{\top}w = 1,\;\; 0 \le w \le c.$$

So Frontier's outer EA only has to evolve the **scalar** `r`; cuOpt solves the
inner QP *exactly* for the weights. Outer search × inner exact solve.

**The honesty check (this notebook's core).** On a convex QP the frontier is
unique, so the EA isn't "discovering" anything an exhaustive ε-sweep wouldn't —
it's *walking* a known curve. The test of correctness is therefore: **does the
EA-walked frontier overlay a direct cuOpt ε-sweep of the same solver?** If yes,
the integration is faithful. The genuine payoff of the EA wrapper shows up only
when you add non-convex structure (cardinality, integers) that breaks the QP —
out of scope here, but the seam is built for it.

**Requirements.** cuOpt runs on **Linux + an NVIDIA GPU only**. In Colab pick a
GPU runtime (*Runtime → Change runtime type → T4 GPU*). Data and the gated
backend live in the Frontier repo, cloned below; the universe is the same 30-ETF
dataset used in Frontier's own evaluations (5-yr history, real returns).

## 1 · Environment

Confirm a GPU is present, then install cuOpt + the Frontier engine.

In [ ]:
import subprocess
try:
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        capture_output=True, text=True,
    )
    print(out.stdout.strip() or "⚠️  nvidia-smi returned nothing.")
except FileNotFoundError:
    print("⚠️  No NVIDIA GPU detected — cuOpt cannot run. "
          "In Colab: Runtime → Change runtime type → GPU (e.g. T4).")

In [ ]:
# Bootstrap: put Frontier's engine on the path, then install cuOpt + engine deps.
# First run ~2-3 min. `frontier` is PRIVATE, so the clone needs a GitHub token:
# add a fine-grained read-only PAT under the 🔑 Secrets panel (left sidebar) named
# GH_TOKEN and grant THIS notebook access. (If the repo is public, no token is
# needed.) The token is read from Colab's secret store — it never appears here.
import os, subprocess, sys

REPO = "/content/frontier"


def _clone() -> int:
    token = ""
    try:
        from google.colab import userdata          # Colab-only
        token = (userdata.get("GH_TOKEN") or "").strip()
    except Exception:
        pass                                        # not on Colab / no secret set
    url = f"https://{token + '@' if token else ''}github.com/cafzal/frontier.git"
    # GIT_TERMINAL_PROMPT=0 → fail fast instead of hanging on a credential prompt.
    return subprocess.run(["git", "clone", "-q", url, REPO],
                          env={**os.environ, "GIT_TERMINAL_PROMPT": "0"}).returncode


if not os.path.isdir(REPO):
    if _clone() != 0 or not os.path.isdir(REPO):
        raise RuntimeError(
            "Could not clone the private `cafzal/frontier` repo. Fix ONE of:\n"
            "  • add a GitHub token as a Colab secret named GH_TOKEN (🔑 sidebar, "
            "grant this notebook access), or\n"
            "  • make the repo public.\n"
            "Then re-run this cell."
        )

# cuOpt from NVIDIA's index + ONLY the engine's runtime deps (not the MCP server),
# so importing the engine stays lean.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "--extra-index-url", "https://pypi.nvidia.com", "cuopt-cu12"],
               check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pymoo>=0.6", "pydantic>=2.0", "scipy>=1.11", "pandas>=2.0",
                "matplotlib>=3.7"], check=True)

if REPO not in sys.path:
    sys.path.insert(0, REPO)
print("Frontier on path:", REPO in sys.path, "| repo present:", os.path.isdir(REPO))


In [ ]:
import json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The one piece that needs the GPU. If this import fails you are not on a
# cuOpt-capable runtime (Linux + NVIDIA GPU).
from cuopt.linear_programming.problem import Problem as CuProblem, MINIMIZE

%config InlineBackend.figure_format = "retina"
plt.rcParams["figure.figsize"] = (8, 5.5)
print("cuOpt import OK — GPU solver is live.")

## 2 · Smoke test — is cuOpt actually solving?

A trivial LP and the 5-asset mean-variance QP from NVIDIA's reference fixture,
each timed. This confirms the GPU path works end-to-end and gives a latency
baseline before we drive hundreds of solves through it. The QP smoke calls the
**exact backend function** Frontier's engine uses (`_solve_qp_cuopt`), so it
tests the real code path, not a throwaway model.

In [ ]:
# --- trivial LP: min 2x0 + 3x1 + x2  s.t. x0+x1+x2 = 1, 0<=xi<=1 ---
p = CuProblem("smoke_lp")
x = [p.addVariable(lb=0.0, ub=1.0, name=f"x{i}") for i in range(3)]
p.setObjective(2 * x[0] + 3 * x[1] + 1 * x[2], sense=MINIMIZE)
p.addConstraint(x[0] + x[1] + x[2] == 1, "sum_to_1")
t = time.time(); p.solve(); dt = (time.time() - t) * 1e3
print(f"LP: status={p.Status}  obj={p.ObjValue:.4f}  ({dt:.1f} ms)   "
      f"expect obj≈1.0 (all weight on cheapest x2)")

In [ ]:
# --- 5-asset QP smoke: NVIDIA's simulated Markowitz fixture (plan P5) ---
from solvers import cuopt_backend as cb

np.random.seed(7)
annual_mean = np.array([0.02, 0.08, 0.075, 0.04, 0.06])
annual_vol  = np.array([0.005, 0.16, 0.18, 0.06, 0.14])
corr = np.array([
    [1.00, 0.05, 0.05, 0.10, 0.05],
    [0.05, 1.00, 0.80, -0.10, 0.55],
    [0.05, 0.80, 1.00, -0.05, 0.50],
    [0.10, -0.10, -0.05, 1.00, 0.00],
    [0.05, 0.55, 0.50, 0.00, 1.00],
])
mcov = np.outer(annual_vol / 12**0.5, annual_vol / 12**0.5) * corr
rets = np.random.multivariate_normal(annual_mean / 12, mcov, size=120)
mu5  = rets.mean(0) * 12
cov5 = np.cov(rets, rowvar=False) * 12

t = time.time()
w_mv, ok = cb._solve_qp_cuopt(cb._nearest_psd(cov5), mu5, None, True, None)
dt = (time.time() - t) * 1e3
print(f"QP min-variance: ok={ok}  ({dt:.1f} ms)")
print("  weights:", np.round(w_mv, 3), " sum=", round(float(w_mv.sum()), 4))
print(f"  return={float(mu5 @ w_mv):+.4f}  risk={float(np.sqrt(w_mv @ cov5 @ w_mv)):.4f}")

## 3 · The problem — a real 30-ETF universe

We model the same universe Frontier uses in its own evaluations: 30 ETFs with
5-year annualized returns and a 30×30 covariance matrix. Two objectives:

| Objective | Direction | Aggregation | Meaning |
|---|---|---|---|
| **Expected Return** | maximize | `sum` (= $\mu^{\top}w$ on the simplex) | portfolio return |
| **Volatility** | minimize | `quadratic` | $\sqrt{w^{\top}\Sigma w}$ — true portfolio risk via covariance |

Allocation is **proportional** (integer percent, summing to 100) with a **30% cap**
per holding. Note the covariance is *indefinite* (min eigenvalue ≈ −83) because
it was built from asset-class correlations rather than a raw sample covariance —
cuOpt's QP needs a PSD `Q`, so the backend projects it onto the PSD cone
(`_nearest_psd`) for the solve while **reporting** risk with the original matrix.

In [ ]:
from engine.models import (
    Aggregation, Direction, InteractionMatrix, Objective, Option, Problem, Score,
)

# Resolve the bundled dataset whether running in Colab or from a local checkout.
_cands = [
    "/content/frontier/examples/cuopt_portfolio/etf_universe.json",
    os.path.join(os.getcwd(), "etf_universe.json"),
    os.path.join(os.getcwd(), "examples/cuopt_portfolio/etf_universe.json"),
]
DATA = next((p for p in _cands if os.path.exists(p)), _cands[0])
data = json.load(open(DATA))

tickers = data["tickers"]
assets  = data["assets"]
cov     = np.array(data["covariance"], float)
mu      = np.array([a["ann_return_5yr_pct"] for a in assets], float)
vols    = np.sqrt(np.diag(cov))
MAX_ALLOC = 30

ev = np.linalg.eigvalsh(0.5 * (cov + cov.T))
print(f"{len(tickers)} ETFs | return {mu.min():.1f}–{mu.max():.1f}% | "
      f"vol {vols.min():.1f}–{vols.max():.1f}% | cov min-eig {ev.min():.1f} (indefinite)")

entries = {a: {b: float(cov[i, j]) for j, b in enumerate(tickers)}
           for i, a in enumerate(tickers)}
problem = Problem(
    name="30-ETF Markowitz (Frontier × cuOpt)",
    approach="proportional",
    objectives=[
        Objective(name="Expected Return", direction=Direction.maximize,
                  unit="annual %", aggregation=Aggregation.sum),
        Objective(name="Volatility", direction=Direction.minimize,
                  unit="annual %", aggregation=Aggregation.quadratic),
    ],
    options=[Option(name=t) for t in tickers],
    scores=[Score(option=t, objective="Expected Return", value=float(mu[i]))
            for i, t in enumerate(tickers)]
    + [Score(option=t, objective="Volatility", value=float(vols[i]))
       for i, t in enumerate(tickers)],
    interaction_matrices=[InteractionMatrix(objective="Volatility", entries=entries)],
    constraints=[{"type": "max_allocation", "max": MAX_ALLOC}],
)
print("Problem built:", problem.name)

## 4 · Walk the frontier — Frontier's EA driving cuOpt

Setting `FRONTIER_SOLVER=cuopt` arms the gate in `optimize()`. The engine routes
this problem (proportional + a quadratic objective) to the cuOpt backend: NSGA-II
evolves the scalar return target, cuOpt solves each min-variance QP on the GPU,
and the results come back in Frontier's **standard `Run`/`Solution` shape** — the
gate is additive and reversible, so every downstream consumer is unaffected.

In [ ]:
os.environ["FRONTIER_SOLVER"] = "cuopt"          # arm the gate
from engine.optimizer import optimize, validate

assert validate(problem).ready, "problem must validate"
t = time.time()
run = optimize(problem, seed=42)
wall = time.time() - t

rows = []
for s in run.solutions:
    rows.append({
        "solution_id": s.solution_id,
        "Volatility": round(s.objective_values["Volatility"], 4),
        "Expected Return": round(s.objective_values["Expected Return"], 4),
        "n_holdings": len(s.selected_options),
        **s.allocations,
    })
df = pd.DataFrame(rows).sort_values("Volatility").reset_index(drop=True)
df.to_csv("cuopt_frontier.csv", index=False)

print(f"{len(run.solutions)} Pareto portfolios in {wall:.2f}s  "
      f"(hypervolume={run.quality.hypervolume_normalized})")
print(f"risk {df['Volatility'].min():.2f}–{df['Volatility'].max():.2f}%   "
      f"return {df['Expected Return'].min():.2f}–{df['Expected Return'].max():.2f}%")
df[["solution_id", "Volatility", "Expected Return", "n_holdings"]].head(8)

## 5 · The efficient frontier

Each point is a cuOpt-optimal portfolio; the curve is the risk/return frontier.

In [ ]:
fig, ax = plt.subplots()
sc = ax.scatter(df["Volatility"], df["Expected Return"], c=df["Expected Return"],
                cmap="viridis", s=70, edgecolor="k", linewidth=0.4, zorder=3)
ax.plot(df["Volatility"], df["Expected Return"], color="gray", lw=1, alpha=0.6, zorder=2)
ax.set_xlabel("Volatility  (annual %, √wᵀΣw)")
ax.set_ylabel("Expected Return  (annual %)")
ax.set_title("Efficient frontier — Frontier EA × cuOpt QP (30 ETFs, 30% cap)")
fig.colorbar(sc, label="Expected Return %")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6 · Correctness — does the EA walk match a direct cuOpt ε-sweep?

This is the honesty check. We bypass the EA entirely and sweep the return target
across its full range, solving the same cuOpt QP at each step, then round and
re-aggregate **identically** to the engine. If the integration is faithful, the
EA-walked points should lie on top of this swept curve. We quantify the gap as
the max nearest-neighbour distance in normalized objective space.

In [ ]:
# Direct ε-sweep with the SAME cuOpt solver the engine used (no EA).
cov_psd = cb._nearest_psd(cov)
mw = MAX_ALLOC / 100.0

prop = cb._opt._ProportionalProblem(
    n_options=len(tickers), score_matrix=cb._opt._build_score_matrix(problem),
    objectives=problem.objectives,
    interaction_matrices=cb._opt._build_interaction_matrices(problem),
    **cb._opt._parse_constraints(problem),
)
ri, rj = cb._resolve_objective_roles(problem)

w_mv, _ = cb._solve_qp_cuopt(cov_psd, mu, None, True, mw)
r_lo, r_hi = float(mu @ w_mv), float(mu.max())

t = time.time()
sweep = []
for tgt in np.linspace(r_lo, r_hi, 250):
    w, ok = cb._solve_qp_cuopt(cov_psd, mu, float(tgt), True, mw)
    if not ok:
        continue
    raw = cb._round_weights_to_pct(w, len(tickers), MAX_ALLOC).astype(float).reshape(1, -1)
    sweep.append((float(prop._aggregate_objective(raw, ri)[0]),
                  float(prop._aggregate_objective(raw, rj)[0])))
sweep = np.array(sorted(set(sweep)))
sweep_wall = time.time() - t

ea = np.array(sorted((d["Volatility"], d["Expected Return"]) for d in rows))
allpts = np.vstack([ea, sweep]); lo, hi = allpts.min(0), allpts.max(0)
span = np.where(hi - lo > 0, hi - lo, 1.0)
max_dev = max(np.min(np.linalg.norm((sweep - lo) / span - p, axis=1)) for p in (ea - lo) / span)
print(f"swept {len(sweep)} cuOpt points in {sweep_wall:.2f}s over return [{r_lo:.2f}, {r_hi:.2f}]")
print(f"max normalized deviation  EA → ε-sweep:  {max_dev:.4f}   "
      f"({'PASS ✓' if max_dev < 0.05 else 'CHECK'}  threshold 0.05)")

In [ ]:
fig, ax = plt.subplots()
ax.plot(sweep[:, 0], sweep[:, 1], "-", color="crimson", lw=2, alpha=0.8,
        label=f"direct cuOpt ε-sweep ({len(sweep)} pts)")
ax.scatter(ea[:, 0], ea[:, 1], s=80, facecolor="none", edgecolor="navy",
           linewidth=1.6, zorder=3, label=f"EA × cuOpt ({len(ea)} pts)")
ax.set_xlabel("Volatility (annual %)"); ax.set_ylabel("Expected Return (annual %)")
ax.set_title(f"EA-walked frontier overlays the exact ε-sweep  (max dev {max_dev:.4f})")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7 · Baseline — Frontier's default NSGA-II (no cuOpt)

The same problem with the gate **off** runs Frontier's stock evolutionary solver,
which evolves full weight vectors directly. Both should trace the same frontier.
The instructive difference: the cuOpt path reaches the frontier's true extremes
exactly (the QP nails the corner portfolios), whereas the stochastic EA often
stops just short of them.

In [ ]:
os.environ.pop("FRONTIER_SOLVER", None)          # disarm the gate → default NSGA-II
t = time.time()
base = optimize(problem, seed=42)
base_wall = time.time() - t
bpts = np.array(sorted((s.objective_values["Volatility"], s.objective_values["Expected Return"])
                       for s in base.solutions))

fig, ax = plt.subplots()
ax.plot(bpts[:, 0], bpts[:, 1], "o", color="darkorange", ms=4, alpha=0.55,
        label=f"NSGA-II baseline ({len(bpts)} pts, {base_wall:.1f}s)")
ax.plot(ea[:, 0], ea[:, 1], "-", color="navy", lw=1.4, alpha=0.6)
ax.scatter(ea[:, 0], ea[:, 1], s=55, color="navy", zorder=3,
           label=f"EA × cuOpt ({len(ea)} pts, {wall:.1f}s)")
ax.set_xlabel("Volatility (annual %)"); ax.set_ylabel("Expected Return (annual %)")
ax.set_title("cuOpt QP vs. stock NSGA-II on the same problem")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"EA×cuOpt   risk [{ea[:,0].min():.2f}, {ea[:,0].max():.2f}]  "
      f"return [{ea[:,1].min():.2f}, {ea[:,1].max():.2f}]")
print(f"NSGA-II    risk [{bpts[:,0].min():.2f}, {bpts[:,0].max():.2f}]  "
      f"return [{bpts[:,1].min():.2f}, {bpts[:,1].max():.2f}]")

## 8 · Frontier → cuOpt: a third objective (dividend yield)

cuOpt solves *one* objective at a time — it has no notion of a tradeoff surface.
Turning exact point-solves into a multi-objective **set** is Frontier's half of the
deal. Here we add a **third** objective, **dividend yield** (already in the dataset),
so the engine walks two epsilon targets (return *and* yield) while cuOpt solves the
same min-variance QP under both floors. The frontier stops being a curve and becomes
a **surface** — return vs risk vs yield — which a single-objective solver structurally
cannot produce.

In [ ]:
# Frontier -> cuOpt: add a 3rd objective (yield). Same Σ/μ, + the dividend-yield column.
yld = np.array([a["dividend_yield_pct"] for a in assets], float)
entries = {a: {b: float(cov[i, j]) for j, b in enumerate(tickers)} for i, a in enumerate(tickers)}
problem3 = Problem(
    name="30-ETF Markowitz + Yield (3-objective)", approach="proportional",
    objectives=[
        Objective(name="Expected Return", direction=Direction.maximize, unit="annual %", aggregation=Aggregation.sum),
        Objective(name="Volatility", direction=Direction.minimize, unit="annual %", aggregation=Aggregation.quadratic),
        Objective(name="Dividend Yield", direction=Direction.maximize, unit="annual %", aggregation=Aggregation.sum),
    ],
    options=[Option(name=t) for t in tickers],
    scores=[Score(option=t, objective="Expected Return", value=float(mu[i])) for i, t in enumerate(tickers)]
        + [Score(option=t, objective="Volatility", value=float(np.sqrt(cov[i, i]))) for i, t in enumerate(tickers)]
        + [Score(option=t, objective="Dividend Yield", value=float(yld[i])) for i, t in enumerate(tickers)],
    interaction_matrices=[InteractionMatrix(objective="Volatility", entries=entries)],
    constraints=[{"type": "max_allocation", "max": MAX_ALLOC}],
)
os.environ["FRONTIER_SOLVER"] = "cuopt"
run3 = optimize(problem3, seed=42)
P = np.array([(s.objective_values["Volatility"], s.objective_values["Expected Return"],
              s.objective_values["Dividend Yield"]) for s in run3.solutions])
print(f"{len(run3.solutions)} portfolios on the 3-objective surface")
print(f"  risk [{P[:,0].min():.2f}, {P[:,0].max():.2f}]  return [{P[:,1].min():.2f}, {P[:,1].max():.2f}]"
      f"  yield [{P[:,2].min():.2f}, {P[:,2].max():.2f}]")

fig = plt.figure(figsize=(8, 6)); ax = fig.add_subplot(projection="3d")
sc = ax.scatter(P[:, 0], P[:, 1], P[:, 2], c=P[:, 2], cmap="plasma", s=55, edgecolor="k", linewidth=0.3)
ax.set_xlabel("Volatility %"); ax.set_ylabel("Return %"); ax.set_zlabel("Yield %")
ax.set_title("Frontier × cuOpt — 3-objective surface (return / risk / yield)")
fig.colorbar(sc, label="Yield %", shrink=0.6); plt.tight_layout(); plt.show()

## 9 · cuOpt → Frontier: exact shadow prices (duals)

Because cuOpt solves each QP *exactly*, it also returns the constraint **duals** —
information a stochastic search (NSGA) structurally cannot produce. The dual of the
return constraint is the **shadow price of return**: the marginal variance you must
accept for one more unit of expected return at that point on the frontier. It is the
local exchange rate that turns "here are the options" into "here is what the next unit
of return costs you" — and on a convex frontier it rises monotonically, certifying you
are on the efficient edge. (`addConstraint` returns the constraint; after `solve()` its
`.DualValue` is populated.)

In [ ]:
# cuOpt -> Frontier: read the dual (shadow price) on the return constraint.
cov_psd = cb._nearest_psd(cov); mw = MAX_ALLOC / 100.0
w_mv, _ = cb._solve_qp_cuopt(cov_psd, mu, None, True, mw)
r_lo, r_hi = float(mu @ w_mv), float(mu.max())
rets, duals = [], []
for tgt in np.linspace(r_lo, r_hi * 0.999, 30):
    n = len(mu); p = CuProblem("dual_demo")
    wv = [p.addVariable(lb=0.0, ub=mw, name=f"w{i}") for i in range(n)]
    quad = None
    for i in range(n):
        for j in range(n):
            c = float(cov_psd[i, j])
            if abs(c) > 1e-12:
                term = c * wv[i] * wv[j]; quad = term if quad is None else quad + term
    p.setObjective(quad, sense=MINIMIZE)
    p.addConstraint(sum(wv) == 1, name="budget")
    rc = p.addConstraint(sum(float(mu[i]) * wv[i] for i in range(n)) >= float(tgt), name="return_target")
    p.solve()
    if getattr(p.Status, "name", "") not in ("Optimal", "PrimalFeasible"):
        continue
    w = np.array([wv[i].Value for i in range(n)])
    rets.append(float(mu @ w)); duals.append(abs(float(rc.DualValue)))

fig, ax = plt.subplots()
ax.plot(rets, duals, "o-", color="purple", lw=1.5)
ax.set_xlabel("Expected Return achieved (annual %)")
ax.set_ylabel("Shadow price  ∂(variance)/∂(return)")
ax.set_title("cuOpt dual — marginal risk cost of return (rises ⇒ efficient frontier)")
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
print(f"shadow price {min(duals):.3f} → {max(duals):.3f} as return target rises — a monotone "
      f"increase certifies the convex efficient frontier, straight from cuOpt's exact dual.")

## 10 · Both directions at once: cardinality ("hold at most K")

This is where neither tool suffices alone. Add a **cardinality** limit — hold at most
**K = 5** of the 30 ETFs — and the clean convex QP breaks: the efficient set now
depends on *which* 5 assets you pick, a combinatorial choice with C(30,5) ≈ 142k
options. **Frontier's EA** searches that discrete space (cuOpt has no concept of it);
**cuOpt** solves the exact min-variance QP on each chosen support (NSGA can't guarantee
the optimum on a fixed support). The EA stops being a redundant frontier-walker and
does real work — encoding the support as a priority vector whose top-K entries are the
eligible assets, with excluded assets pinned to a 0 upper bound (so it stays inside
cuOpt's continuous-QP beta — no MIQP).

Three checks: the cardinality frontier (it sits *inside* the unconstrained one — fewer
names, less diversification), a value comparison vs Frontier's stock NSGA under the same
cap, and a correctness check by **brute-force support enumeration** on a small
sub-universe (the combinatorial analog of the ε-sweep).

In [ ]:
# Both directions: cardinality. EA picks which-K; cuOpt solves the exact QP on that support.
problem_card = Problem(
    name="30-ETF Markowitz, hold ≤ 5", approach="proportional",
    objectives=problem.objectives, options=problem.options, scores=problem.scores,
    interaction_matrices=problem.interaction_matrices,
    constraints=[{"type": "max_allocation", "max": MAX_ALLOC}, {"type": "cardinality", "min": 1, "max": 5}],
)
os.environ["FRONTIER_SOLVER"] = "cuopt"
run_card = optimize(problem_card, seed=42)
cardpts = np.array(sorted((s.objective_values["Volatility"], s.objective_values["Expected Return"])
                          for s in run_card.solutions))
max_hold = max(len(s.selected_options) for s in run_card.solutions)
print(f"{len(run_card.solutions)} cardinality portfolios; max holdings = {max_hold} (cap 5)")

fig, ax = plt.subplots()
ax.plot(df["Volatility"], df["Expected Return"], "-", color="lightgray", lw=4, alpha=0.8,
        label="unconstrained (≤30 names)")
ax.scatter(cardpts[:, 0], cardpts[:, 1], s=60, color="crimson", edgecolor="k", linewidth=0.4,
           zorder=3, label="hold ≤ 5 (EA × cuOpt)")
ax.set_xlabel("Volatility %"); ax.set_ylabel("Expected Return %")
ax.set_title("Cardinality frontier sits inside the unconstrained one")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# Value: same cap, Frontier's stock NSGA (gate off) — cuOpt reaches exact corners NSGA misses.
os.environ.pop("FRONTIER_SOLVER", None)
base_card = optimize(problem_card, seed=42)
os.environ["FRONTIER_SOLVER"] = "cuopt"
bc = np.array(sorted((s.objective_values["Volatility"], s.objective_values["Expected Return"])
                     for s in base_card.solutions))
print(f"NSGA (cap 5): {len(base_card.solutions)} pts, "
      f"max holdings {max(len(s.selected_options) for s in base_card.solutions)}")

fig, ax = plt.subplots()
ax.scatter(bc[:, 0], bc[:, 1], s=45, color="darkorange", alpha=0.7, label=f"stock NSGA, ≤5 ({len(bc)} pts)")
ax.scatter(cardpts[:, 0], cardpts[:, 1], s=55, facecolor="none", edgecolor="navy", linewidth=1.5,
           zorder=3, label=f"EA × cuOpt, ≤5 ({len(cardpts)} pts)")
ax.set_xlabel("Volatility %"); ax.set_ylabel("Expected Return %")
ax.set_title("Under a hold-≤5 cap: exact QP-on-support vs stochastic NSGA")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# Correctness: brute-force support enumeration on a small sub-universe (tractable
# exhaustive check) = the combinatorial analog of §6's ε-sweep overlay.
from itertools import combinations
sub = np.argsort(mu)[np.linspace(0, len(mu) - 1, 8).astype(int)]   # 8 assets spanning the return range
mus, covs = mu[sub], cov[np.ix_(sub, sub)]
cps = cb._nearest_psd(covs); mw = MAX_ALLOC / 100.0
Ksub = 4   # ≥ ceil(100/MAX_ALLOC)=4, else 3×30%<100% is infeasible
bf = []
for combo in combinations(range(8), Ksub):                          # every C(8,Ksub) support
    wmv, ok = cb._solve_qp_cuopt(cps, mus, None, True, mw, support=combo)
    if not ok:
        continue
    lo, hi = float(mus @ wmv), float(mus[list(combo)].max())
    for t in np.linspace(lo, hi, 12):
        w, ok = cb._solve_qp_cuopt(cps, mus, float(t), True, mw, support=combo)
        if ok:
            bf.append((float(np.sqrt(max(w @ covs @ w, 0.0))), float(mus @ w)))
bf = np.array(sorted(set(bf)))

subtk = [tickers[i] for i in sub]
sub_entries = {a: {b: float(covs[i, j]) for j, b in enumerate(subtk)} for i, a in enumerate(subtk)}
problem_sub = Problem(
    name="sub-universe ≤3", approach="proportional", objectives=problem.objectives,
    options=[Option(name=t) for t in subtk],
    scores=[Score(option=t, objective="Expected Return", value=float(mus[i])) for i, t in enumerate(subtk)]
        + [Score(option=t, objective="Volatility", value=float(np.sqrt(covs[i, i]))) for i, t in enumerate(subtk)],
    interaction_matrices=[InteractionMatrix(objective="Volatility", entries=sub_entries)],
    constraints=[{"type": "max_allocation", "max": MAX_ALLOC}, {"type": "cardinality", "min": 1, "max": Ksub}],
)
run_sub = optimize(problem_sub, seed=42)
eapts = np.array(sorted((s.objective_values["Volatility"], s.objective_values["Expected Return"])
                        for s in run_sub.solutions))
allp = np.vstack([eapts, bf]); lo, hi = allp.min(0), allp.max(0); span = np.where(hi - lo > 0, hi - lo, 1.0)
dev = max(np.min(np.linalg.norm((bf - lo) / span - q, axis=1)) for q in (eapts - lo) / span)

fig, ax = plt.subplots()
ax.plot(bf[:, 0], bf[:, 1], ".", color="crimson", ms=4, alpha=0.5, label=f"brute force C(8,{Ksub}) ({len(bf)} pts)")
ax.scatter(eapts[:, 0], eapts[:, 1], s=70, facecolor="none", edgecolor="navy", linewidth=1.6, zorder=3,
           label=f"EA × cuOpt ≤{Ksub} ({len(eapts)} pts)")
ax.set_xlabel("Volatility %"); ax.set_ylabel("Expected Return %")
ax.set_title(f"Cardinality correctness: EA recovers the brute-force envelope (max dev {dev:.3f})")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
print(f"max normalized deviation EA -> brute-force: {dev:.4f}  ({'PASS' if dev < 0.10 else 'CHECK'} < 0.10 "
      f"— combinatorial bar, looser than §6's convex 0.05 by design)")

## 11 · Findings — the integration is bidirectional

The convex demo (§4–§7) proves the *plumbing* is faithful. §8–§10 prove the
**mutual** value — each tool does something the other structurally cannot:

- **Frontier → cuOpt (a frontier, not a point).** cuOpt is single-objective; §8 turns
  its exact solves into a 2- and 3-objective **surface** (return / risk / yield). The
  multi-objective elicitation and Pareto exploration are Frontier's to give.
- **cuOpt → Frontier (exactness + diagnostics).** §9 reads cuOpt's **dual** — the
  shadow price of return, the marginal risk cost of one more unit of return — which a
  metaheuristic cannot produce. Exact corners and provable optimality are cuOpt's to give.
- **Both at once (cardinality).** §10 adds "hold ≤ 5": the convex QP breaks, a plain
  ε-sweep no longer suffices, and the EA does real combinatorial search over *which*
  assets while cuOpt solves each support exactly. The EA earns its keep here; cuOpt
  guarantees each inner solve. The brute-force overlay confirms correctness.

**The thesis, demonstrated:** *cuOpt makes each point exact and fast; Frontier makes the
set a decision.* Neither the 3-objective surface, the shadow prices, nor the exact
cardinality frontier exists with either tool alone — that is the integration's value,
and it is additive and reversible (one env var flips the whole cuOpt path on or off).

The `cuopt_frontier.csv` written in §4 holds the base convex frontier; the cells above
add the cardinality, 3-objective, and dual results.